# CFP Ranker '26

## Data Collection

### Team Collection

In [1]:
%pip install requests
%pip install pandas
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# NOTE: This block of code extracts the list of FBS teams from ESPN's API and saves it to a CSV file.
# This is arguably more useful because it includes team IDs and conference information
import requests
import pandas as pd

url = "https://site.web.api.espn.com/apis/v2/sports/football/college-football/standings?group=80"
# 80 is the ID for the FBS (Football Bowl Subdivision) in ESPN's API

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

data = response.json()

def parse_standings_node(node, parent_conference="", parent_conf_id=""):
    """Recursively walks standings nodes to extract teams, conference names, and conference IDs."""
    teams = []
    
    current_name = node.get('name', '')
    current_id = str(node.get('id', ''))
    
    # Prevent top-level league/subdivision labels from overriding actual conference names/IDs
    is_root_label = current_name.lower() in ['fbs', 'ncaa - football bowl subdivision', 'football bowl subdivision']
    
    if is_root_label:
        conf_label = ""
        conf_id = ""
    else:
        conf_label = parent_conference if parent_conference else current_name
        conf_id = parent_conf_id if parent_conf_id else current_id

    # Extract teams if standings entries exist at this level
    entries = node.get('standings', {}).get('entries', [])
    for entry in entries:
        team = entry.get('team', {})
        if team:
            teams.append({
                "team_id": team.get('id'),
                "display_name": team.get('displayName'),
                "short_name": team.get('shortDisplayName'),
                "conference": conf_label,
                "conference_id": conf_id
            })

    # Recurse into sub-children (e.g. East/West divisions)
    for child in node.get('children', []):
        teams.extend(parse_standings_node(child, parent_conference=conf_label, parent_conf_id=conf_id))

    return teams

fbs_teams = parse_standings_node(data)

df = pd.DataFrame(fbs_teams)
df = df.drop_duplicates(subset=['team_id']).reset_index(drop=True)

df.to_csv("espn_fbs_teams.csv", index=False)
print(f"Successfully extracted {len(df)} FBS teams with conference IDs.")

Successfully extracted 138 FBS teams with conference IDs.


### Schedule Parsing

In [4]:
import json
import requests
import pandas as pd
import time
from datetime import datetime

# --- CONFIG ---
YEAR = 2026  # Works for any year (e.g., 2023, 2024, 2025, 2026)
FBS_TEAMS_FILE = "../data/espn_fbs_teams.csv"

# --- ENDPOINTS & HEADERS ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event="

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json"
}
